# Looking at the resolVI embedding of the ULMS tumor cells only
- Subsetting resolVI for tumor cells - manually annotated based off the scVIVA embedding, now mapping back to resolVI embedding
- Did not retrain a tumor-only model since resolVI needs spatial information, so just recalcuated the neighbors graph and UMAP from the original embedding

In [12]:
import sys
import numpy as np
import scanpy as sc
import pandas as pd
import anndata as ad
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.sparse as sp
import re

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpascvi

In [3]:
# version control
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)

plt.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['pdf.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
mpl.rcParams['ps.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
plt.ioff()
sc.settings.autoshow = False
sc.settings.n_jobs = -1  # Use all available cores

[rank: 0] Seed set to 1234


pandas: 2.3.1
numpy: 2.2.6
scanpy: 1.11.4
scvi: 1.3.3


In [6]:
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent
print(PARENT_DIR)

SCVIVA_ANN_DIR = PARENT_DIR / 'scviva_tumor'

output_dir = jpascvi.create_output_dir(PARENT_DIR, 'resolvi_tumor', change_dir=True)

/oak/stanford/groups/longaker/ULMS/revision/G4X
Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/resolvi_tumor
Working directory and scanpy figure output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/resolvi_tumor
DATA_DIR is: /oak/stanford/groups/longaker/ULMS/revision/G4X/resolvi


In [7]:
jpa_markers = jpascvi.import_markers((PARENT_DIR / 'ref/jpa_g4x_breast_panel.csv'), output_type='dict')
jpa_markers = {key: value for key, value in jpa_markers.items() if key != 'Plasma_cell'} # JCHAIN and IGHG1 not in this segmentation run
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

{'Adipocyte': ['LPL', 'ADIPOQ'], 'B_cell': ['MS4A1', 'CD79A', 'CD19'], 'DC': ['CD1C', 'CLEC9A', 'HLA-DRA', 'GPR183'], 'EC': ['PECAM1', 'VWF', 'ACKR1', 'LYVE1'], 'Fibroblast': ['FAP', 'LUM', 'DPT', 'COL1A1', 'FN1', 'THY1', 'MGP', 'POSTN'], 'Macrophage': ['CD163', 'CD68', 'TLR2', 'CSF1R', 'IDO1'], 'Mast_cell': ['KIT'], 'Monocyte': ['FCGR3A', 'CD14', 'S100A9', 'S100A10', 'TLR4', 'WARS1', 'STAT1'], 'Neutrophil': ['CD33', 'CD44', 'CEACAM8', 'VEGFA'], 'NK_cell': ['GNLY', 'FCGR3A', 'NCAM1', 'KLRB1', 'KLRF1', 'KLRD1', 'NKG7', 'GZMA', 'GZMB', 'GZMH'], 'Pericyte': ['RGS5', 'PDGFRB', 'CSPG4'], 'Plasma_cell': ['CD38', 'SDC1', 'CD79A', 'MZB1', 'JCHAIN', 'TNFRSF17', 'IGHA1', 'IGHD', 'IGHG1', 'IGHM'], 'Proliferation': ['MKI67', 'TOP2A', 'RRM2'], 'Smooth_muscle': ['DES', 'ESR1', 'PGR', 'ACTA2', 'TAGLN', 'MYH11', 'ACTG2', 'AR', 'MYL9', 'MYLK', 'TNC'], 'T_cell': ['CD2', 'CD3D', 'CD3E', 'IL7R', 'CD247', 'CD4', 'CD8A']}


# Import data

In [9]:
# load in the annotated scVIVA object for the tumor subset
print(f"Loading annotated scVIVA object for the tumor subset from {SCVIVA_ANN_DIR}")
adata = sc.read_h5ad(SCVIVA_ANN_DIR / 'tumor_annotated.h5ad')
print(adata)

AnnData object with n_obs × n_vars = 1597386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', 'true_proportion', 'diffusion_proportion', 'background_proportion', 'coarse_celltype', 'index', '_scvi_sample', 'scviva_coarse_ct', 'celltype', 'ann_leiden', 'leiden0_1', 'leiden0_2', 'leiden0_3', 'leiden0_4', 'leiden0_5', 'leiden0_6', 'leiden0_7', 'leiden0_8', 'leiden0_9', 'leiden1_0', 'leiden1_1', 'leiden1_2', 'leiden1_3', 'leiden1_4', 'leiden1_5', 'leiden1_6', 'leiden1_7', 'leiden1_8', 'leiden1_9', 'leiden2_0', 'tumor_subtype'
    var: 'mean', 'std'
    uns: 'Patient_colors', 'Section_colors', 'ann_leiden_colors', 'celltype_colors', 'coarse_celltype_colors', 'dendrogram_leiden0_1', 'dendrogram_leiden0_2', 'dendrogram_leiden0_3', 'dendrogram_leiden0_4', 'dendrogram_le

In [13]:
# Clean up
# Drop obs columns matching patterns
pattern = re.compile(r'^(leiden\d|_scvi)')
adata.obs = adata.obs[[c for c in adata.obs.columns if not pattern.match(c)]]

# Drop uns keys matching patterns
uns_pattern = re.compile(r'^(leiden\d|dendrogram_leiden|_scvi)')
static_drops = {'log1p', 'neighbors', 'rank_genes_groups', 'umap'}
for key in list(adata.uns.keys()):
    if uns_pattern.match(key) or key in static_drops:
        del adata.uns[key]

del adata.obsp['connectivities']
del adata.obsp['distances']
del adata.obsm['X_umap']

print(adata)

AnnData object with n_obs × n_vars = 1983386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', '_indices', 'true_proportion', 'diffusion_proportion', 'background_proportion'
    var: 'mean', 'std'
    uns: 'Patient_colors', 'Section_colors'
    obsm: 'X_resolVI', 'X_spatial', 'X_umap', 'distance_neighbor', 'index_neighbor'
    layers: 'X_normalized_resolVI', 'counts', 'generated_expression'
    obsp: 'connectivities', 'distances'


In [ ]:
# Make sure to use the X_resolVI embedding
sc.pp.neighbors(adata, use_rep="X_resolVI")
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
# saving the anndata now that umap has been computed
adata.write_h5ad(output_dir / 'resolvi_tumor.h5ad')

# Clustering and Differential Expression

In [ ]:
sc.pl.umap(adata, color=['Section', 'Patient'], frameon=False, save='section_and_patient.png')
sc.pl.umap(adata, color=['component', 'volume', 
                         'surface_area', 'n_genes_by_counts', 
                         'log1p_n_genes_by_counts', 'total_counts', 
                         'log1p_total_counts', 'n_counts', 'n_genes',], frameon=False, save='qc.png')

In [ ]:
jpascvi.featureplot(adata, jpa_markers, save='resolvi.png', use_raw=False, vmax='p98') # False to allow scaled values to be plotted from adata.X
jpascvi.featureplot(adata, jpa_markers, save='corr_resolvi.png', layer="generated_expression", vmax='p98')
jpascvi.featureplot(adata, jpa_markers, save='norm_resolvi.png', layer="X_normalized_resolVI", vmax='p98')

In [ ]:
markers = ['MYH11', 'MYLK', 'DES', 'TAGLN', 'ESR1', 'COL1A1', 'SDC1', 'SOX10']
sc.pl.umap(adata, color=markers, size=0.2, save='ulms_feature_plot.png', vmax='p98')

markers = ['VEGFA', 'SLC2A1']
sc.pl.umap(adata, color=markers, size=0.2, save='ischemia_feature_plot.png', vmax='p98')

markers = ['VWF', 'PECAM1', 'COL1A1', 'LUM', 'CD68', 'CD163', 'RGS5', 'PDGFRB', 'CD2', 'IL7R']
sc.pl.umap(adata, color=markers, size=0.2, save='nontumor_feature_plot.png', vmax='p98')

markers = ['ESR1', 'PGR', 'AR']
sc.pl.umap(adata, color=markers, save='hormononal_feature_plot.png', size=0.2, vmax='p98')

markers = ['LAG3', 'PDCD1']
sc.pl.umap(adata, color=markers, save='immune_checkpoint_feature_plot.png', size=0.2, vmax='p98')

In [ ]:
# Leiden clustering at multiple resolutions and differential gene expression
for resolution in resolutions:
    str_res = str(resolution).replace('.', '_')
    leiden_key = 'leiden' + str_res
    sc.tl.leiden(adata, flavor="igraph", n_iterations=2, resolution=resolution, key_added=leiden_key)
    jpascvi.plot_umap(adata, resolution)
    jpascvi.sc_degs(adata, resolution, use_rep='X_resolVI', plots=['dotplot'])

# Calculate clustering statistics to see which is the optimal resolution from a mathematical perspective
jpascvi.cluster_stats(adata, resolutions, scores = ['Calinski-Harabasz', 'Davies-Bouldin'], rep='X_resolVI')

In [ ]:
adata.write_h5ad(output_dir / 'resolvi_tumor_clustered.h5ad')